# 30-Day Data Engineer Python + Pandas Plan

This notebook is built for a senior data engineer profile: Python, pandas, SQL-style transformations, pipeline logic, performance thinking, and production mindset.

Use it as a working practice notebook:
- each section maps to a day or small set of days
- prompts are data-engineering focused, not analyst-focused
- each exercise includes a scaling question
- many sections include pandas to Spark translation notes

## What Data Engineering Interviews Actually Test

- data transformations
- SQL thinking in Python
- pipeline design
- data quality and edge cases
- performance at scale
- production reasoning

Keep asking:
> How would this behave on millions of rows?

And:
> How would I productionize this in Spark / AWS?

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import Counter, defaultdict

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## Shared Practice Data

In [ ]:
raw_users = pd.DataFrame([
    {"user_id": "U1", "name": " Alice ", "city": "New York", "signup_date": "2026-01-01", "status": "active"},
    {"user_id": "U2", "name": "bob", "city": " chicago ", "signup_date": "2026-01-02", "status": "active"},
    {"user_id": "U3", "name": None, "city": "Seattle", "signup_date": "2026-01-03", "status": "inactive"},
    {"user_id": "U2", "name": "Bob", "city": "Chicago", "signup_date": "2026-01-02", "status": "active"},
])

transactions = pd.DataFrame([
    {"txn_id": 1, "user_id": "U1", "amount": 100, "txn_ts": "2026-01-01 10:00:00", "country": "US", "updated_at": "2026-01-01 10:01:00"},
    {"txn_id": 2, "user_id": "U2", "amount": 50, "txn_ts": "2026-01-01 10:30:00", "country": "US", "updated_at": "2026-01-01 10:31:00"},
    {"txn_id": 3, "user_id": "U1", "amount": 70, "txn_ts": "2026-01-01 11:00:00", "country": "US", "updated_at": "2026-01-01 11:05:00"},
    {"txn_id": 4, "user_id": "U3", "amount": None, "txn_ts": "2026-01-02 09:00:00", "country": "CA", "updated_at": "2026-01-02 09:01:00"},
    {"txn_id": 5, "user_id": "U2", "amount": 120, "txn_ts": "2026-01-02 10:00:00", "country": "US", "updated_at": "2026-01-02 10:03:00"},
    {"txn_id": 5, "user_id": "U2", "amount": 125, "txn_ts": "2026-01-02 10:00:00", "country": "US", "updated_at": "2026-01-02 10:10:00"},
    {"txn_id": 6, "user_id": "U4", "amount": -10, "txn_ts": "2026-01-03 08:00:00", "country": "US", "updated_at": "2026-01-03 08:01:00"},
])

events = pd.DataFrame([
    {"user_id": "U1", "event_ts": "2026-01-01 10:00:00", "event_type": "login"},
    {"user_id": "U1", "event_ts": "2026-01-01 10:05:00", "event_type": "view"},
    {"user_id": "U1", "event_ts": "2026-01-01 11:00:00", "event_type": "purchase"},
    {"user_id": "U2", "event_ts": "2026-01-01 09:00:00", "event_type": "login"},
    {"user_id": "U2", "event_ts": "2026-01-01 09:20:00", "event_type": "view"},
    {"user_id": "U2", "event_ts": "2026-01-01 10:10:00", "event_type": "logout"},
    {"user_id": "U3", "event_ts": "2026-01-02 12:00:00", "event_type": "login"},
    {"user_id": "U3", "event_ts": "2026-01-02 12:50:00", "event_type": "view"},
])

transactions["txn_ts"] = pd.to_datetime(transactions["txn_ts"])
transactions["updated_at"] = pd.to_datetime(transactions["updated_at"])
raw_users["signup_date"] = pd.to_datetime(raw_users["signup_date"])
events["event_ts"] = pd.to_datetime(events["event_ts"])

raw_users, transactions.head(), events.head()

# Week 1: Data Manipulation + Real Data Thinking

## Day 1-2: Data Exploration Like Production Debugging

Goal:
- inspect shape, schema, nulls, duplicates, bad values
- identify what would break downstream

Questions:
- What data quality issues do you see in `raw_users`?
- What data quality issues do you see in `transactions`?
- Which columns look unsafe for downstream joins, aggregations, or partitioning?
- Which issues are blocking vs non-blocking for a production pipeline?

Tasks:
- inspect `raw_users`
- inspect `transactions`
- write down at least 5 data issues you see

In [ ]:
# Explore raw_users and transactions here
# Suggested:
# raw_users.shape
# raw_users.dtypes
# raw_users.isnull().sum()
# transactions.describe(include='all')

# TODO

Scaling question:
- If this were a daily landing table in S3, which checks would you automate before loading to curated storage?

## Day 3-4: Data Cleaning

Questions:
- How should `name` and `city` be normalized for consistent analytics?
- Which columns need explicit type conversion?
- How should null names and null amounts be handled?
- Which cleaning rules belong in code vs metadata/config?

Tasks:
- normalize `name`
- normalize `city`
- convert types
- decide how to handle null names and null amounts

Output:
- `clean_users`
- `clean_transactions`

In [ ]:
def clean_user_data(df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError


def clean_transaction_data(df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

Spark mapping:
- pandas `str.strip().str.lower()` -> Spark `trim(lower(col(...)))`
- pandas `fillna` -> Spark `fillna`

Production question:
- Which cleaning decisions should be configurable vs hardcoded?

## Day 5-6: Dedup + Data Integrity

Real DE pattern:
- keep latest transaction record per `txn_id` using `updated_at`
- identify duplicate `user_id` rows in `raw_users`

Questions:
- What is the correct business key for transactions?
- Which timestamp should decide the winning record?
- How should ties be handled if `updated_at` is identical?
- Should duplicate users be merged, dropped, or quarantined?

Tasks:
- create `latest_transactions`
- create `deduped_users`
- document assumptions

In [ ]:
def latest_record_per_key(df: pd.DataFrame, key_cols, ts_col: str) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## Day 7: Mini Pipeline

Questions:
- What are the logical stages of a small but production-style pipeline?
- What should each stage consume and produce?
- Where would you place validation and logging?
- What outputs would be useful for debugging failed runs?

Build a mini pipeline:
- raw users -> cleaned users -> deduped users
- raw transactions -> cleaned transactions -> latest transactions

Return a dict of stage outputs.

In [ ]:
def build_week1_pipeline(raw_users_df: pd.DataFrame, raw_txn_df: pd.DataFrame):
    # TODO
    raise NotImplementedError

# Week 2: Aggregations + SQL Thinking

## Day 8-9: GroupBy Core DE Skill

Questions:
- How do you compute daily revenue from valid transactions?
- How do you compute user-level spend and transaction counts?
- Which rows should be excluded before aggregation?
- What would you validate after computing the aggregates?

Using latest valid transactions only:
- daily revenue
- total revenue per user
- transaction count per user

In [ ]:
def daily_revenue(df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError


def user_metrics(df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## Day 10-11: Multi-Level Aggregation

Questions:
- How do you aggregate by `country` and `txn_date`?
- How do you aggregate by `user_id` and `txn_date`?
- What schema should the output tables have?
- Which grouping dimensions would become partitions in a data lake?

Compute metrics by:
- `country`, `txn_date`
- `user_id`, `txn_date`

Include:
- total amount
- avg amount
- txn count

In [ ]:
def multi_level_aggregations(df: pd.DataFrame):
    # TODO
    raise NotImplementedError

## Day 12-13: Transform

Questions:
- How do you add user-level totals back to row-level data?
- When should you use `transform` instead of `agg` + merge?
- What are common mistakes when mixing row-level and aggregated data?
- How would this logic translate to Spark window functions?

Add these columns to transactions:
- `user_total_amount`
- `user_avg_amount`
- `country_total_amount`

Use `groupby(...).transform(...)`.

In [ ]:
def add_transform_metrics(df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## Day 14: SQL to pandas Mapping

Questions:
- What is the pandas equivalent of `GROUP BY`?
- What is the pandas equivalent of `LEFT JOIN`?
- How do you simulate `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` in pandas?
- Which SQL patterns become awkward or expensive in pandas?

Implement the pandas equivalent of:
- GROUP BY
- LEFT JOIN
- ROW_NUMBER partitioned by user ordered by timestamp desc

Use transactions and users as input.

In [ ]:
# Write your SQL-to-pandas examples here

# Week 3: Joins + Window + Pipeline Logic

## Day 15-16: Joins

Questions:
- How do you join fact transactions to user dimensions?
- How do you detect missing keys in the dimension table?
- When should you use inner vs left joins in a pipeline?
- What join metrics would you log in production?

Practice:
- fact + dimension join: transactions with users
- identify missing dimension keys
- compare inner vs left join behavior

In [ ]:
def join_transactions_users(txn_df: pd.DataFrame, users_df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError


def find_missing_dimension_keys(txn_df: pd.DataFrame, users_df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## Day 17-18: Window Functions

Questions:
- How do you rank transactions within each user?
- How do you compute running totals in timestamp order?
- How do you retrieve the previous transaction amount per user?
- What ordering bugs can break window calculations?

Add:
- rank of each transaction by amount within user
- running total by user ordered by timestamp
- previous transaction amount per user

In [ ]:
def add_window_metrics(df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

Spark mapping:
- pandas `cumsum` -> Spark window + `sum().over(...)`
- pandas `rank` -> Spark `dense_rank` / `row_number`

## Day 19-20: Top N per Group

Questions:
- How do you return the top N transactions per user?
- How should ties be handled?
- How would you generalize this to top N per country per day?
- What is the Spark equivalent of this pattern?

Return top 2 transactions per user by amount.

Questions:
- How do you break ties?
- What if you need top N per day per country?

In [ ]:
def top_n_transactions_per_user(df: pd.DataFrame, n: int = 2) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## Day 21: Build a Data Pipeline

Questions:
- What stages would you define for a metrics pipeline?
- Which outputs belong in curated tables vs monitoring tables?
- Where should anomaly detection happen?
- How would you make the pipeline idempotent?

Pipeline target:
- cleaned transactions
- user-level metrics
- daily metrics
- top users
- anomaly candidates

Define clear stage boundaries.

In [ ]:
def build_metrics_pipeline(users_df: pd.DataFrame, txn_df: pd.DataFrame):
    # TODO
    raise NotImplementedError

# Week 4: Real Data Engineering Problems

## Day 22-23: Sessionization

Questions:
- How do you define a session boundary?
- How do you assign session IDs per user?
- How do you produce session start, end, and event count?
- What changes when this becomes a streaming problem?

A new session starts if the gap between consecutive events for the same user exceeds 30 minutes.

Return:
- `user_id`
- `session_id`
- `session_start`
- `session_end`
- `event_count`

In [ ]:
def sessionize(events_df: pd.DataFrame, session_gap_minutes: int = 30) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## Day 24-25: Incremental Processing

Questions:
- How do you merge new transactions into an existing latest-state table?
- How do you guarantee that only the newest version wins?
- What should happen to unchanged historical rows?
- How would this differ with CDC events or late-arriving updates?

Simulate new data arriving.

Requirements:
- merge new transactions into existing latest transactions
- keep latest by `txn_id` and `updated_at`
- produce updated user aggregates

In [ ]:
new_transactions = pd.DataFrame([
    {"txn_id": 2, "user_id": "U2", "amount": 55, "txn_ts": "2026-01-01 10:30:00", "country": "US", "updated_at": "2026-01-01 10:50:00"},
    {"txn_id": 7, "user_id": "U1", "amount": 90, "txn_ts": "2026-01-03 12:10:00", "country": "US", "updated_at": "2026-01-03 12:12:00"},
])
new_transactions["txn_ts"] = pd.to_datetime(new_transactions["txn_ts"])
new_transactions["updated_at"] = pd.to_datetime(new_transactions["updated_at"])

def incremental_update(existing_df: pd.DataFrame, incoming_df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## Day 26-27: Data Quality Checks

Questions:
- Which null checks are mandatory for this dataset?
- How do you detect duplicate keys?
- How do you detect invalid negative amounts?
- How do you detect missing foreign keys from transactions to users?
- What should the output report look like?

Build checks for:
- null thresholds
- duplicate keys
- invalid negative amounts
- missing dimension keys

Return a structured report DataFrame.

In [ ]:
def run_data_quality_checks(users_df: pd.DataFrame, txn_df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## Day 28: Performance Thinking

Questions:
- Why is `.apply()` often a red flag in pandas interview solutions?
- Which operations here can be vectorized?
- When would you use chunk processing?
- At what point should this move from pandas to Spark?

Write brief answers or small examples for:
- why `.apply()` is often a bad sign
- where vectorization helps
- how chunk processing works
- when pandas is the wrong tool

In [ ]:
# Write notes or code examples here

## Day 29: Mock Interview Tasks

Questions:
- How do you compute daily spend per city after joining transactions to users?
- How do you rank cities by revenue per day?
- How do you detect whether a user's latest transaction is anomalously high?
- Which edge cases would you call out before writing code?

Solve both:

1. Join transactions with users, compute daily spend per city, and rank cities by revenue per day.
2. Detect users whose latest transaction amount is greater than 2x their historical average.

Think about edge cases before coding.

In [ ]:
# Mock interview workspace

## Day 30: System Thinking

Questions:
- How would you scale one of these pandas solutions to Spark?
- How would you productionize it on AWS?
- What orchestration layer would you use and why?
- How would you test correctness, performance, and recovery behavior?
- How would you monitor data failures and drift over time?

For one solution from this notebook, explain:
- how it would scale to Spark
- how you would productionize it on AWS
- what orchestration you would use
- how you would test it
- how you would monitor failures and data drift

In [ ]:
# Write your production/system design answer here

# Must-Know DE Patterns

- latest record per key
- top N per group
- running totals
- fact + dimension joins
- sessionization
- data cleaning pipelines

# Pandas to Spark Mapping

In [ ]:
mapping = pd.DataFrame([
    {"pandas": "groupby", "spark": "groupBy"},
    {"pandas": "merge", "spark": "join"},
    {"pandas": "cumsum", "spark": "window + sum over"},
    {"pandas": "rank", "spark": "dense_rank / row_number"},
    {"pandas": "apply", "spark": "UDF (avoid if possible)"},
])
mapping

# Daily Routine

- 20 min concept
- 40 min coding
- 10 min scaling discussion

Do not practice like a beginner. Favor pipeline thinking, data correctness, and scale-aware reasoning.

# Validation Sandbox

In [ ]:
# Example calls after you implement functions:
# clean_user_data(raw_users)
# clean_transaction_data(transactions)
# latest_record_per_key(transactions, ['txn_id'], 'updated_at')
# build_week1_pipeline(raw_users, transactions)
# daily_revenue(transactions)
# user_metrics(transactions)
# multi_level_aggregations(transactions)
# add_transform_metrics(transactions)
# join_transactions_users(transactions, raw_users)
# add_window_metrics(transactions)
# top_n_transactions_per_user(transactions)
# sessionize(events)
# incremental_update(transactions, new_transactions)
# run_data_quality_checks(raw_users, transactions)